# Phase 7 — Évaluation corrigée + significativité statistique

Éval corrigée (vertu avec trait, utilitarisme contre-balancé, tirage seedé, balance de classes) puis tests McNemar / bootstrap / Wilson.

## ⚠️ Datasets à attacher
- `adl-dpo-adapter` (notebook 01)
- `adl-rlhf-model-fixed` (notebook 06)

**Durée** : ~30-40 min.

> Prérequis : avoir poussé `eval/evaluate_ethics_fixed.py` et `eval/stats_analysis.py`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q -U bitsandbytes transformers==4.46.3 trl==0.12.0 peft==0.14.0 \
    accelerate==1.2.0 datasets==3.2.0 sentence-transformers faiss-cpu \
    anthropic openai pyarrow==17.0.0 tqdm

In [ ]:
!rm -rf /kaggle/working/adl
!git clone https://github.com/FeelTheFloww/adl-ethics.git /kaggle/working/adl
%cd /kaggle/working/adl

In [ ]:
# Vérifie que les scripts corrigés sont bien dans le repo cloné.
# Si ça échoue : commit + push des 4 fichiers sur ton repo GitHub d'abord.
import os
for p in ["eval/kl_diagnostics.py","training/train_ppo_fixed.py",
          "eval/evaluate_ethics_fixed.py","eval/stats_analysis.py"]:
    assert os.path.isfile(p), f"MANQUANT: {p} — pousse les scripts corrigés sur GitHub."
print("Scripts corrigés présents.")

In [ ]:
DPO_ZIP  = "/kaggle/input/adl-dpo-adapter/dpo_model.zip"
RLHF_ZIP = "/kaggle/input/adl-rlhf-model-fixed/rlhf_model_fixed.zip"
import os, zipfile
os.makedirs("results/dpo_model", exist_ok=True)
os.makedirs("results/rlhf_model_fixed", exist_ok=True)
with zipfile.ZipFile(DPO_ZIP)  as z: z.extractall("results/dpo_model")
with zipfile.ZipFile(RLHF_ZIP) as z: z.extractall("results/rlhf_model_fixed")
print("DPO:",  os.listdir("results/dpo_model"))
print("RLHF:", os.listdir("results/rlhf_model_fixed"))

In [ ]:
!python eval/evaluate_ethics_fixed.py \
    --base_model     Qwen/Qwen2.5-1.5B-Instruct \
    --dpo_adapter    results/dpo_model \
    --rlhf_adapter   results/rlhf_model_fixed \
    --ethical_corpus data/ethical_corpus_v2.json \
    --output_path    results/eval_results_fixed.json \
    --n_per_cat      100 --seed 0

In [ ]:
# Significativité : McNemar apparié + bootstrap + Wilson CI (aucun GPU requis)
!python eval/stats_analysis.py --results results/eval_results_fixed.json

## Figure KL corrigée (optionnel)

Mets à jour la légende : la KL devrait être ~0 à l'init (π_θ = π_ref).

In [ ]:
!python eval/plot_ppo_kl.py \
    --trainer_state results/rlhf_model_fixed/checkpoint-125/trainer_state.json \
    --out results/ppo_kl_fixed.png || echo "(trainer_state.json non trouvé, skip)"

In [ ]:
import json
print(json.dumps(json.load(open("results/eval_results_fixed.json")), indent=2)[:2000])